# One check, taken apart

The reconciliation loop closes the difference between what should exist and
what does. Everything it ever does is one **check**, three steps long:

| step | question | reads | writes |
|---|---|---|---|
| **observe** | is the thing intent implies actually in storage? | S3, at one address | `materialized_models` |
| **gap** | what should exist, minus what does | a snapshot | nothing — it is a pure function |
| **act** | close the difference | — | submits a job, or records why not |

Two ideas carry the whole design, and this notebook shows each one happening:

**Intent implies an address.** A model's identity is a hash of the inputs that
define it — methodology commit, reach geometry, DEM source, and so on. All of
those live in the database, so the loop can compute where a model *must* be
before any job runs. Observation is a lookup, never a search.

**Results flow upstream, so work flows downstream-first.** A reach's model
needs its downstream neighbour's model *and* ND library (the max-q stage
transfer line shapes this reach's geometry). Terminal reaches — outlets — have
no downstream, so a fresh network starts building at its outlets and everything
else waits its turn.

**Before running:** `docker compose up -d db minio minio-init`, and the
`build_model` image available locally.

In [ ]:
import json
import time

import pandas as pd

from recon import activity, check, db, gap, identity, intent, jobs, observe, processing, queue, storage
from recon.config import settings
from recon.workers import LocalDockerRunner

pd.set_option("display.max_colwidth", 42)

print(f"database  {settings.postgres_host}:{settings.postgres_port}/{settings.postgres_db}")
print(f"storage   s3://{settings.artifacts_s3_bucket} via {settings.aws_endpoint_url}")
print(f"job image {settings.build_model_image}")

## 1. Author intent

Nothing exists yet, and nothing is asked for. Three inserts change that:

- **`desired_state_defaults`** — one row, the deployment's intent: the six
  identity inputs (methodology commit, grid resolution, EPSG, DEM source, LULC
  source and lookup) plus fallbacks for everything else. These values must
  match what the deployed job image would use, because the loop predicts
  addresses from them. `sdr_commit` in particular is baked into the image and
  cannot be overridden — the row records it, the image enforces it.
- **`reach_network`** — the topology, from the hydrofabric.
- **`desired_state`** — one row per reach, all fields NULL: *"I want this
  reach, defaults for everything."* A reach in the network means nothing until
  this row exists.

We also clear the models prefix in storage so the story starts from zero —
deleting from storage is the supported way to undo work, so this is an
ordinary operation, not surgery.

In [ ]:
import sys
sys.path.insert(0, "../scripts")
from seed import load_network

LULC_LOOKUP = {"11":0.04,"21":0.04,"22":0.1,"23":0.08,"24":0.15,"31":0.025,"41":0.16,
               "42":0.16,"43":0.16,"52":0.1,"71":0.035,"81":0.03,"82":0.035,"90":0.12,"95":0.07}

network = load_network("../testdata/network.gpkg")

with db.connect() as conn:
    conn.execute("TRUNCATE reach_network CASCADE")
    conn.execute("DELETE FROM desired_state_defaults")
    conn.execute(
        """INSERT INTO desired_state_defaults
           (sdr_commit, grid_resolution, epsg_code, dem_source, lulc_source, lulc_lookup)
           VALUES (%s, 10, 5070, %s, %s, %s)""",
        ("826a602ddcaf58bf4081dc04b65ba15b82cc8c8a",
         "https://prd-tnm.s3.amazonaws.com/StagedProducts/Elevation/13/TIFF/USGS_Seamless_DEM_13.vrt",
         "/data/Annual_NLCD_LndCov_2023_CU_C1V0.tif",
         json.dumps(LULC_LOOKUP)))
    for r in network:
        reason = r["terminal_reason"] or ("outlet" if r["is_terminal"] else None)
        conn.execute(
            """INSERT INTO reach_network (reach_id, reach_to_id, is_terminal, is_headwater,
                 terminal_reason, total_da_sqkm, stream_order, slope, geom)
               VALUES (%s,%s,%s,%s,%s,%s,%s,%s, ST_GeomFromText(%s, 5070))""",
            (r["reach_id"], r["reach_to_id"], r["is_terminal"], r["is_headwater"],
             reason, r["total_da_sqkm"], r["stream_order"], r["slope"], r["geom"]))
    conn.execute("INSERT INTO desired_state (reach_id) SELECT reach_id FROM reach_network")

# start storage from zero as well
s3 = storage.get_s3_client()
bucket, prefix = storage.parse_s3_path(f"s3://{settings.artifacts_s3_bucket}/version=v{settings.major_version}/models")
for page in s3.get_paginator("list_objects_v2").paginate(Bucket=bucket, Prefix=prefix):
    for obj in page.get("Contents", []):
        s3.delete_object(Bucket=bucket, Key=obj["Key"])

print(db.one("SELECT count(*) AS reaches, count(*) FILTER (WHERE is_terminal) AS terminals FROM reach_network"))
pd.DataFrame(db.table_counts())

## 2. The queue is a question

There is no queue data structure. Asking the database *which reaches need
looking at* **is** the queue — which is why a reconciler that dies mid-sweep
loses nothing. Each row says why it is due.

In [ ]:
due = pd.DataFrame(queue.due_reaches())
print(f"{len(due)} reaches due")
due.head(6)

## 3. Effective intent, and the address it implies

Pick two reaches: a **terminal** (an outlet — no downstream) and a
**non-terminal** just above one. For each, the loop resolves effective intent
(`COALESCE(desired_state.x, desired_state_defaults.x)`), builds the identity
object the job would build, and hashes it.

That hash is the address. No job has run, yet we know exactly where each
reach's model must appear in the bucket.

In [ ]:
rows = db.query("""
    SELECT rn.reach_id, rn.is_terminal FROM reach_network rn
    JOIN reach_network ds ON ds.reach_id = rn.reach_to_id
    WHERE ds.is_terminal LIMIT 1""")
upstream_id = rows[0]["reach_id"]
terminal_id = db.one("SELECT reach_to_id AS r FROM reach_network WHERE reach_id=%s", (upstream_id,))["r"]
print(f"terminal reach:     {terminal_id}")
print(f"non-terminal above: {upstream_id}\n")

wanted = intent.effective(terminal_id)
identity_obj, identity_hash = identity.model_identity(wanted)
print("identity object the job will build:")
print(json.dumps(identity_obj, indent=2))
print(f"\npredicted identity hash: {identity_hash}")
print(f"predicted address:       {storage.model_base_path(terminal_id)}/{identity_hash}_<domain>/")

## 4. Observe — a lookup, not a search

Observe looks at that one address. Anything else in the bucket — models from
older intent, a neighbour's artifacts — is invisible to it, so there is
nothing to rank and no "newest wins".

Nothing is there yet, so no proof row is written.

In [ ]:
print("observe:", observe.observe_reach(terminal_id))
print("proof rows:", db.query("SELECT * FROM materialized_models"))

## 5. Gap — a pure function decides

The snapshot is one query: this reach's proofs, its downstream neighbour's
proofs, and whether a job is already in flight. `gap.calculate` is pure — no
database, no storage, no clock — so the same snapshot always gives the same
answer, and you can read every rule the loop has in one file.

Watch the ladder treat the two reaches differently: the terminal may build;
the non-terminal must wait, and the decision says for whom.

In [ ]:
for rid, label in ((terminal_id, "terminal"), (upstream_id, "non-terminal")):
    snap = check.load_snapshot(rid)
    print(f"{label} {rid}:")
    print(f"   snapshot: model_ok={snap.model_ok} ds_model_ok={snap.ds_model_ok} ds_nd_ok={snap.ds_nd_ok}")
    print(f"   decision: {gap.calculate(snap)}\n")

## 6. Act — submit and walk away

`run_check` performs all three steps and acts on the decision. For the
terminal that means submitting a real `build_model` container — and returning
immediately. The fact that work is running lives in the **database**
(`current_step`, with the container id as the handle), not in this notebook's
memory: kill the kernel now and nothing is lost.

Checking again while the job runs does not resubmit — that is the in-flight
marker doing its job. Checking the non-terminal records who it waits for.

In [ ]:
import os
env_vars = {"AWS_ENDPOINT_URL": "http://minio:9000"}
for key in ("AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_SESSION_TOKEN", "AWS_REQUEST_PAYER"):
    if os.environ.get(key):
        env_vars[key] = os.environ[key]
runner = LocalDockerRunner(
    image=settings.build_model_image, network=settings.docker_network, env_vars=env_vars,
    platform=settings.docker_platform,
    volumes=[f"{settings.docker_data_dir}:/data:ro"] if settings.docker_data_dir else [])

print("terminal:    ", check.run_check(terminal_id, runner))
print("check again: ", check.run_check(terminal_id, runner), "   <- no resubmit")
print("non-terminal:", check.run_check(upstream_id, runner))
print()
print(pd.DataFrame(processing.in_flight()))

## 7. Hear back

The **job status pass** asks the execution system what became of the jobs the
database says are in flight. It records nothing about what exists — it only
clears the marker and requests a check. Whether anything was *produced* is
storage's question, answered by the next observe.

In [ ]:
deadline = time.time() + 900
while time.time() < deadline:
    outcomes = jobs.status_pass(runner)
    if not outcomes:
        print("nothing in flight")
        break
    print(f"{time.strftime('%H:%M:%S')}  {outcomes[0]['status']:<10} {outcomes[0]['action']}")
    if outcomes[0]["status"] in ("succeeded", "failed"):
        break
    time.sleep(15)

## 8. The next check finds the proof

Observe looks at the predicted address again — and now the manifest is there.
Before adopting it, the loop verifies it: the manifest must belong to this
reach, sit in the folder its hash names, carry exactly the identity fields the
loop knows, and its identity object must re-hash to the value it claims. Only
then is the proof row written, stamped with the revision it proves.

The number to check: the job computed its identity **independently, inside the
container** — and landed on the hash we predicted in step 3.

In [ ]:
print(check.run_check(terminal_id, runner))
print()
row = db.one("SELECT * FROM materialized_models WHERE reach_id=%s", (terminal_id,))
print("proof row: ", row)
print(f"\npredicted {identity_hash} == adopted {row['identity_hash']}:", identity_hash == row["identity_hash"])

## 9. The ladder holds

The non-terminal is still waiting — its downstream now has a **model**, but
the geometry transfer also needs the downstream **ND library**, which no job
can produce yet (run_nd_scenarios is the next milestone). The whole network
therefore settles at: terminals finished, everyone else waiting, and the state
of every reach derived — never stored — by `reach_status`.

In [ ]:
print("non-terminal now:", check.run_check(upstream_id, runner))
print()
print(pd.DataFrame(db.query(
    "SELECT state, count(*) AS reaches FROM reach_status GROUP BY state ORDER BY reaches DESC")))
print()
pd.DataFrame(activity.recent(8, reach_id=terminal_id))[["action", "outcome", "revision", "detail"]]

## What to remember

- **A check is short and never waits.** Three checks saw one job through:
  submit, leave alone, adopt. A crash between any two costs nothing.
- **Intent implies the address.** The job and the loop computed the same hash
  from the same inputs without talking to each other — that is what lets
  observation be a single lookup, and what makes two models in one bucket
  unambiguous.
- **The row is proof.** `materialized_models` has a row exactly when a reach's
  model intent is satisfied, stamped with the revision it proves. Deleting the
  model from storage deletes the row, and the claim with it.
- **Dependencies gate work, not proof.** The ladder decides what may *start*;
  what already exists at the right address is adopted no matter what.

`02_cases.ipynb` runs the awkward situations — deletions, tampering, intent
changes, failures. `03_run_network.ipynb` lets the loop run the whole network.